# On the information bottleneck theory for neural networks
## Final Report
### Giacomo Menegatti, Dario Puggioni

## Abstract
Neural networks are used ubiquitously in Machine Learning, but their apllication still lacks a theoretical basis, which reflects in a lack of knowledge about choosing the right architecture, which is usually chosen empirically as the one which performs better than the others over a set of test samples.

The **information bottleneck** theory tries to explain the learning process as an information problem by studying the mutual information between the network internal (or _hidden_) variables and the input and output variables, and predicts the appearance of specific patterns inside of the network as a good metric to classify its performance.

In this work we show our results regarding the information recovered by the network over two image based datasets, MNIST and CIFAR10, and try to offer some answers to the questions posed in the two papers on the topic, _Opening the Black Box of Deep Neural Networks via Information_ by _Shwartz-Ziv & Tishby_ and _ON THE INFORMATION BOTTLENECK THEORY OF DEEP LEARNING_ by _Saxe & al_; in particular:

- does the mutual information show a _learning_ and a _compressing_ phase?
- is it dependent on the architecture of the network?
- Is it dependent on the chosen activation function?

## Mutual information estimators

### The Mutual information

The model computation task is $Y \leftarrow X $ $ \rightarrow H_1 \rightarrow H_2 \rightarrow ... \rightarrow \hat Y $, where $X$ is the input, $Y$ the true labels and $\hat Y$ the reconstructed label predicted by the network, and $H_i = f_i(W_iH_{i-1})$ the hidden variables at the i-th layer.

The mutual information is defined as the mean information for a distribution $X$ that can be reconstructed from another distribution $Y$: given the Shannon entropy $H_S[X]=-\sum_i p_i \log p_i$, the mutual information is $I(X:Y)=H[X,Y]- H[X] - H[Y]=H[X]-H[X|Y]$.

In this analysis we reconstruct $I(X:H_i)$ and $I(Y:H_i)$, which are the mutual information contained in the hidden variables w.r.t the input and the true labels distributions (NOT the predicted $\hat Y$, as they do not represent the true scope of the task). The expectations are that the learning process will maximize $I(Y:H)$. 

We will refer to the loss of mutual information as _compression_. In these examples, a loss of $I(X:H)$ is expected to be beneficial as the network is discarding some information about the inputs which is unneccessary for inferring the label. Instead, a loss of $I(H:Y)$ indicates that the network is getting worse at performing its task, which could likely be attribuited to overfitting.

In the first case, $I(X:H_i) = H(H_i) - H(H_i|X) = H(H_i)$ because the variable $H_i$ _are deterministically derived_ by the inputs $X$ thanks to neural network.

In the second, $I(Y:H_i) = H(H_i) - H(H_i|Y)$ as $Y$ and $H_i$ are not deterministically linked. So to compute $H(H_i|Y)$ first the conditional probability must be computed (as $H(X|Y) = -\sum p(x,y)log(\frac{p(x,y)}{p(y)}) $). In the discretized case, this means that after binning $H_i$ for each input data $x_i$ ypu have to compute the corresponding bin $H_i$ and the corresponding label $y_i$, then computing the marginal distributions as $p(H_i|y_i)$


### The Estimators

The mutual information of a hidden activity variable with the inputs or outputs is measured by a MI estimator, to have a finite entropy (as $h$ is a continuous variable with a definite function). In this report we analysed the use of three estimators.

### Binning estimator
The distribution can be approximated using finite bins. After getting the number of samples in each bin as $p_i = \frac{n_i}{N}$, the entropy is computed directly. This suffers from the curse of dimensionality: the total number of bins scales as $N^d$, where $N$ is the number of bins in each dimensions and $d$ the number of dimensions. To avoid having most bins empty, the Saxe approach is to instead discretive the vectors using `numpy.digitize` to discretize the value each component of the vector can assume, then counting the number of unique vectors. This avoids creating empty bins, but can still degenerate to a uniform distribution when most bins contains only one vector. In this case the entropy is $h=\log N_{points}$. 

### The Kozachenk-Leonenko estimator / KSG estimator

This method uses a similar estimate to KDE. The probability around a point is estimated as $p(x_i)\approx c_d \varepsilon (i)^d$, where $c_d$ is the volume of the unit d-sphere in d dimensions, $\varepsilon(i)$ the distance to the i-th nearest neighbour. From this and the order statistic $E(\log p_i) = \psi(k) -\psi(N)$, the entropy is found as $H(x)= - \psi(k) + \psi(N) + \log c_d + \frac{d}{N}\sum_i^N \log \varepsilon(i)$, where $\psi$ is the _digamma_ function. A similar implementation is given for the mutual information as the **Kraskov-Stögbauer-Grassberger** estimator, which computes the same thing but avoids errors when comparing the single distributions and the joint distribution (which have a different number of dimensions). 

The $k$ parameter controls the smoothness of the reconstructed distribution: $k=1$ measures only the nearest neighbour and so emphatizes peaks and small scales features, while $k=3$ measures the third nearest neighbour, averaging the distribution over further points. In literature it is suggested to use $k \in [3,10]$, or usually $k=4$.

### The Kolchinsky upper and lower bound
This estimator considers the underlying pdf as a mixture of gaussians centered around each point. The mutual information between the hidden variables and the input is given by $$ I(T;X) \leq -\frac{1}{P}\sum_i \log (\frac{1}{P}\sum_j \exp(-\frac{||h_i - h_j||^2}{2\sigma^2}))$$
and w.r.t the real labels Y is $$ I(T;Y) \leq -\frac{1}{P}\sum_i \log (\frac{1}{P}\sum_j \exp(-\frac{||h_i - h_j||^2}{2\sigma^2}))- \sum_l p_l[-\frac{1}{P_l}\sum_{i:Y_i=l} \log \frac{1}{P_l}\sum_{j, Y_l=j}\exp(-\frac{||h_i - h_j||^2}{2\sigma^2})]$$

To this mutual information there is also a lower bound given by $$ I(T;Y) \geq -\frac{1}{P}\sum_i \log (\frac{1}{P}\sum_j \exp(-\frac{||h_i - h_j||^2}{8\sigma^2}))- \sum_l p_l[-\frac{1}{P_l}\sum_{i:Y_i=l} \log \frac{1}{P_l}\sum_{j, Y_l=j}\exp(-\frac{||h_i - h_j||^2}{8\sigma^2})]$$

The KLE depends on the gaussians variance $\sigma^2$, that is fixed by the user.
Both binning and KLE suffer from the curse of dimensionality, as we have around $N=10k$ vectors with $d=1024$, and degenerate to $\log N$.

## The results

To obtain these results, we trained the neural networks until convergence over the respective datasets. During the _testing phase_, when the network computes loss and accuracy over samples _which it has not seen before_, we saved the hidden variables $h_i$ for each layer and input, then computed its entropy and the conditional entropy w.r.t. the true labels.

### MNIST dataset, 1024-20-20-20 FFNN
We trained a fully connected feed forward neural network to convergence over the MNIST dataset, then analysed the mutual information with the Kolchinsky upper and lower bound.

As expected from the _Saxe_ paper, the $\tanh$ presents a compression phase that the $ReLU$ does not have, as the $\tanh$ compressed an input from $(-\infty, +\infty)$ to $(-1,1)$, while the $ReLU$ output is unbound $(0, +\infty)$.

As expected by the papers, each subsequent layer contains less information about the input.

| Loss and accuracy, $ReLU$ activation | Loss and accuracy, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/loss_1024-20-20-20-relu.png) | ![Figure](plots/loss_1024-20-20-20-tanh.png) |

| Kolchinsky upper and lower bound to I(X:H), $ReLU$ activation | Kolchinsky upper and lower bound to I(X:H), $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/MNIST_relu_I(X,T).png) | ![Figure](plots/MNIST_tanh_I(X,T).png) |

| Information plane, $ReLU$ activation | Information plane, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/MNIST_relu_infplane_epochs_upper.png) | ![Figure](plots/MNIST_tanh_infplane_epochs_upper.png) |

| Information plane, $ReLU$ activation | Information plane, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/MNIST_relu_infplane_layers_upper.png) | ![Figure](plots/MNIST_tanh_infplane_layers_upper.png) |

### MNIST DATASET, Convolutional Neural Network
In this part we trained a 2-layer CNN connected to a 3-layer FNN on the MNIST dataset. Due to hardware contraints, we were not able to compute the mutual information at the Convolution layers (which is $N_{samples}\times N_{channels} \times image\_ size$), we computed the mutual informationonly for the final _Global Pool_ layer and the subsequent FFNN layers.

From the results we observe different pattern in the information plane compared to the previous. Only the last layer saturates $I(H:Y)$ and the layers are reversed w.r.t $I(X:H)$, also some of the intermediate layers show a compression of $I(H:Y)$, which means they are losing what wuold be expected to be important information for the task. 

Also there seems to be much less of a difference between the two activation functions.

| Loss and accuracy, $ReLU$ activation | Loss and accuracy, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/loss_MNIST_relu_cnn(16-32)_256_64.png) | ![Figure](plots/loss_MNIST_tanh_cnn(16-32)_256_64.png) |

| Kolchinsky upper and lower bound to I(X:H), $ReLU$ activation | Kolchinsky upper and lower bound to I(X:H), $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/MNIST_cnn_relu_I(X,T).png) | ![Figure](plots/MNIST_cnn_tanh_I(X,T).png) |

| Information plane, $ReLU$ activation | Information plane, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/MNIST_cnn_relu_infplane_epochs_upper.png) | ![Figure](plots/MNIST_cnn_tanh_infplane_epochs_upper.png) |

| Information plane, $ReLU$ activation | Information plane, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/MNIST_cnn_relu_infplane_layers_upper.png) | ![Figure](plots/MNIST_cnn_tanh_infplane_layers_upper.png) |

### CIFAR10 dataset, Convolutional neural network

| Loss and accuracy, $ReLU$ activation | Loss and accuracy, $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/loss_CIFAR10_relu_CNN(16-32)_256_64.png.jpeg) | ![Figure](plots/loss_CIFAR10_tanh_CNN_(16_32)_256_64.png.jpeg) |

| Kolchinsky upper and lower bound to I(X:H), $ReLU$ activation | Kolchinsky upper and lower bound to I(X:H), $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/CIFAR10_relu_I(X,T).png) | ![Figure](plots/CIFAR10_tanh_I(X,T).png) |

| Kolchinsky upper and lower bound to I(X:H), $ReLU$ activation | Kolchinsky upper and lower bound to I(X:H), $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/CIFAR10_relu_infplane_epochs_upper.png) | ![Figure](plots/CIFAR10_tanh_infplane_epochs_upper.png) |

| Kolchinsky upper and lower bound to I(X:H), $ReLU$ activation | Kolchinsky upper and lower bound to I(X:H), $\tanh$ activation |
|:---:|:---:|
| ![Figure](plots/CIFAR10_relu_infplane_layers_upper.png) | ![Figure](plots/CIFAR10_tanh_infplane_layers_upper.png) |

From the graphs it appears that the learning is highly affected by the activation function chosen, much more than in the previous example. In the $ReLU$ case, the _Global Pool_ is learning next to nothing and so is the first connected layer, while the last layer is recovering most of the mutual information both w.r.t $X$ and $\hat Y$, while in the $\tanh$ case all three layers have a similar behaviour. 

Also, in this case the subsequent layers appear to _recover_ more information about the inputs, both in the $\tanh$ and in the $ReLU$ case. Only the _Global Pool_ layer shows some form of compression during training, but this is not shared by the following networks.

## Conclusions
While more test would be necessary to support our findings, from these we can confidently refute some of the findings of the proposed papers, i.e. the presence of a clear pattern in the information plane during the learning process and the presence of a consistent _input compression phase_, and also that the difference in the activation functions is not enough to completely explain it.